# Deepfake Training Notebook

This notebook reorganizes the training script into readable, runnable sections for configuration, datasets, model setup, training, and evaluation.

## 1. Imports and setup

Load all dependencies used by the training pipeline.

In [1]:
!pip install tqdm

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import random
from typing import Any, cast

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import yaml
from IPython.display import clear_output, display
from PIL import Image
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, random_split
from torchvision.models import efficientnet_b0
from tqdm import tqdm

plt.style.use('seaborn-v0_8-whitegrid')

## 2. Transform fallbacks and utilities

These helpers keep the notebook compatible across slightly different `torchvision` environments and provide reusable metric/config utilities.

In [3]:
class Identity(object):
    def __init__(self, *args, **kwargs):
        pass

    def __call__(self, x):
        return x


if hasattr(transforms, 'Resize'):
    Resize = transforms.Resize
else:
    class Resize(object):
        def __init__(self, size):
            self.size = tuple(size) if isinstance(size, (tuple, list)) else (size, size)

        def __call__(self, img):
            return img.resize(self.size, Image.BILINEAR)


RandomHorizontalFlip = getattr(transforms, 'RandomHorizontalFlip', Identity)
RandomRotation = getattr(transforms, 'RandomRotation', Identity)
ColorJitter = getattr(transforms, 'ColorJitter', Identity)
ToTensor = getattr(transforms, 'ToTensor')
Normalize = getattr(transforms, 'Normalize')


def compute_metrics(outputs, labels):
    """Compute batch-level precision, recall, and F1 for the binary real-vs-fake classifier."""
    preds = torch.argmax(outputs, dim=1)
    true_pos = ((preds == 1) & (labels == 1)).sum().item()
    false_pos = ((preds == 1) & (labels == 0)).sum().item()
    false_neg = ((preds == 0) & (labels == 1)).sum().item()

    precision = true_pos / (true_pos + false_pos + 1e-8)
    recall = true_pos / (true_pos + false_neg + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return precision, recall, f1


def load_config(config_path='E:/Verifixia/Verifixia/Backend/pytorch/config.yaml'):
    """Load training settings from a YAML config file."""
    config_path = str(config_path)
    candidate_paths = [config_path]
    if not os.path.isabs(config_path):
        candidate_paths.append(os.path.join(os.getcwd(), config_path))
        candidate_paths.append(os.path.abspath(config_path))

    for path in candidate_paths:
        if path and os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                config = yaml.safe_load(f)
            config['config_path'] = os.path.abspath(path)
            return config

    raise FileNotFoundError(
        f"Config file not found. Tried:\n" + "\n".join(p for p in candidate_paths if p)
    )


def set_seed(seed=42):
    """Set seeds for reproducible dataset splitting and training behavior where possible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


<>:40: SyntaxWarning: invalid escape sequence '\B'
<>:40: SyntaxWarning: invalid escape sequence '\B'
C:\Users\adiup\AppData\Local\Temp\ipykernel_19304\3064735672.py:40: SyntaxWarning: invalid escape sequence '\B'
  def load_config(config_path='Verifixia\Backend\pytorch\config.yaml'):


## 3. Dataset helpers

Support both image-based and video-based deepfake datasets.

In [4]:
def find_class_dir(base_path, class_name, prefer_video_dirs=False):
    """Locate the most likely directory for a class label, with optional preference for video folders."""
    candidates = []
    for entry in os.listdir(base_path):
        entry_path = os.path.join(base_path, entry)
        if os.path.isdir(entry_path) and class_name in entry.lower():
            candidates.append((entry, entry_path))

    if prefer_video_dirs:
        for entry, path in candidates:
            if 'video' in entry.lower() or 'vedio' in entry.lower():
                return path
    else:
        for entry, path in candidates:
            if 'video' not in entry.lower() and 'vedio' not in entry.lower():
                return path

    exact = os.path.join(base_path, class_name)
    if os.path.exists(exact):
        return exact
    return candidates[0][1] if candidates else exact


class ImageDeepfakeDataset(torch.utils.data.Dataset):
    """Dataset for still images organized into real/fake class folders."""
    def __init__(self, data_path, transform=None):
        self.data_path = data_path
        self.transform = transform
        self.samples = []
        for label, class_name in enumerate(['real', 'fake']):
            class_dir = find_class_dir(data_path, class_name, prefer_video_dirs=False)
            if os.path.exists(class_dir):
                for fname in os.listdir(class_dir):
                    if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                        self.samples.append((os.path.join(class_dir, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


class VideoDeepfakeDataset(torch.utils.data.Dataset):
    """Dataset for videos that converts each clip into a fixed-length sequence of frames."""
    def __init__(self, data_path, frames_per_video=12, transform=None):
        self.data_path = data_path
        self.frames_per_video = frames_per_video
        self.transform = transform
        self.samples = []
        for label, class_name in enumerate(['real', 'fake']):
            class_dir = find_class_dir(data_path, class_name, prefer_video_dirs=True)
            if os.path.exists(class_dir):
                for fname in os.listdir(class_dir):
                    if any(fname.lower().endswith(ext) for ext in ['.mp4', '.avi', '.mov', '.mkv', '.webm']):
                        self.samples.append((os.path.join(class_dir, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        frames = self._extract_frames(video_path)
        if self.transform:
            frames = [self.transform(frame) for frame in frames]
        else:
            frames = [ToTensor()(frame) for frame in frames]
        return torch.stack(frames), label

    def _extract_frames(self, video_path):
        """Sample evenly spaced frames so every video produces a fixed-length sequence for the LSTM."""
        cap = cv2.VideoCapture(video_path)
        frames = []
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            cap.release()
            dummy = Image.new('RGB', (224, 224))
            return [dummy] * self.frames_per_video

        indices = [int(i) for i in torch.linspace(0, total_frames - 1, self.frames_per_video).long()]
        for i in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(Image.fromarray(frame))
            else:
                frames.append(frames[-1] if frames else Image.new('RGB', (224, 224)))

        cap.release()
        while len(frames) < self.frames_per_video:
            frames.append(frames[-1])
        return frames[:self.frames_per_video]


## 4. Model definition

Use EfficientNet-B0 as the backbone, replace SiLU with ReLU, and apply an LSTM for temporal modeling on video frames.

In [5]:
def replace_activations_with_relu(module):
    """Recursively replace EfficientNet SiLU activations with ReLU to mirror the project's earlier CNN design."""
    for name, child in module.named_children():
        if isinstance(child, nn.SiLU):
            setattr(module, name, nn.ReLU(inplace=True))
        else:
            replace_activations_with_relu(child)


class DeepfakeCNNLSTM(nn.Module):
    """Hybrid deepfake classifier built from an EfficientNet-B0 feature extractor and an LSTM temporal head."""

    """
    Architecture overview:
    - Backbone: EfficientNet-B0 pretrained on ImageNet, used as a 1280-dim feature extractor.
    - Activation policy: all SiLU activations are replaced with ReLU for consistency with the earlier custom CNN setup.
    - Temporal modeling: for video input, frame-level features are passed through a 2-layer LSTM.
    - Classification: the final hidden state is regularized with dropout and mapped to 
um_classes.

    Input behavior:
    - Video mode expects shape (batch, frames, channels, height, width).
    - Image mode expects shape (batch, channels, height, width) and skips the LSTM.
    """
    def __init__(self, dataset_type='video', num_classes=2, frames_per_video=12):
        super().__init__()
        self.dataset_type = dataset_type
        self.frames_per_video = frames_per_video

        self.backbone = efficientnet_b0(weights='IMAGENET1K_V1')
        self.backbone.classifier = nn.Identity()
        replace_activations_with_relu(self.backbone)

        self.lstm = nn.LSTM(
            input_size=1280,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        """Run a forward pass for either frame sequences or single images."""

        """
        In video mode, frames are flattened into a batch, encoded independently by EfficientNet,
        reshaped back into a sequence, and summarized by the LSTM's last hidden state.
        In image mode, the EfficientNet features are sent directly to the classifier head.
        """
        if self.dataset_type == 'video':
            b, f, c, h, w = x.shape
            x = x.view(b * f, c, h, w)
            feat = self.backbone(x)
            feat = feat.view(b, f, 1280)
            _, (hidden, _) = self.lstm(feat)
            return self.classifier(hidden[-1])

        feat = self.backbone(x)
        return self.classifier(feat)


## 5. Data transforms

Training uses augmentation, while validation applies only deterministic preprocessing.

In [6]:
def get_transforms(config, is_train=True):
    """Build preprocessing pipelines for training or validation data."""

    """
    Training mode applies spatial and color augmentation before tensor conversion and normalization.
    Validation mode keeps preprocessing deterministic so metrics reflect model behavior rather than random transforms.
    Both branches resize samples to the configured input size expected by the EfficientNet backbone.
    """
    aug = config['augmentation']
    size = aug.get('resize', 224)
    if is_train:
        return transforms.Compose([
            Resize((size, size)),
            RandomHorizontalFlip(),
            RandomRotation(aug.get('random_rotation', 15)),
            ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            ToTensor(),
            Normalize(mean=aug['normalize_mean'], std=aug['normalize_std']),
        ])

    return transforms.Compose([
        Resize((size, size)),
        ToTensor(),
        Normalize(mean=aug['normalize_mean'], std=aug['normalize_std']),
    ])

def initialize_history():
    """Create a container for epoch-by-epoch metrics shown in the live training charts."""
    return {
        'epoch': [],
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'precision': [],
        'recall': [],
        'f1': [],
    }


def update_history(history, epoch, train_loss, train_acc, val_loss=None, val_acc=None, precision=None, recall=None, f1=None):
    """Append one epoch of metrics to the in-memory history used for plotting and reporting."""
    history['epoch'].append(epoch)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(np.nan if val_loss is None else val_loss)
    history['val_acc'].append(np.nan if val_acc is None else val_acc)
    history['precision'].append(np.nan if precision is None else precision)
    history['recall'].append(np.nan if recall is None else recall)
    history['f1'].append(np.nan if f1 is None else f1)


def display_training_dashboard(history, config, has_validation):
    """Render live plots so training progress can be monitored inside the notebook after each epoch."""
    clear_output(wait=True)
    last_epoch = history['epoch'][-1]
    print(f'Training progress: epoch {last_epoch}/{config["num_epochs"]}')

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

    axes[0].plot(history['epoch'], history['train_loss'], marker='o', linewidth=2, label='Train Loss')
    if has_validation and not np.isnan(history['val_loss']).all():
        axes[0].plot(history['epoch'], history['val_loss'], marker='o', linewidth=2, label='Val Loss')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

    axes[1].plot(history['epoch'], history['train_acc'], marker='o', linewidth=2, label='Train Acc')
    if has_validation and not np.isnan(history['val_acc']).all():
        axes[1].plot(history['epoch'], history['val_acc'], marker='o', linewidth=2, label='Val Acc')
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()

    if has_validation:
        axes[2].plot(history['epoch'], history['precision'], marker='o', linewidth=2, label='Precision')
        axes[2].plot(history['epoch'], history['recall'], marker='o', linewidth=2, label='Recall')
        axes[2].plot(history['epoch'], history['f1'], marker='o', linewidth=2, label='F1')
        axes[2].set_ylabel('Score')
    else:
        axes[2].plot(history['epoch'], history['train_acc'], marker='o', linewidth=2, label='Train Acc')
        axes[2].set_ylabel('Accuracy (%)')
    axes[2].set_title('Validation Metrics' if has_validation else 'Training Metric')
    axes[2].set_xlabel('Epoch')
    axes[2].legend()

    fig.tight_layout()
    display(fig)
    plt.close(fig)


def print_epoch_summary(epoch, config, train_loss, train_acc, val_loss=None, val_acc=None, precision=None, recall=None, f1=None):
    """Print a compact epoch summary under the live plots for quick reading while training runs."""
    summary = [
        f'Epoch: {epoch}/{config["num_epochs"]}',
        f'Train Loss: {train_loss:.4f}',
        f'Train Acc: {train_acc:.2f}%',
    ]
    if val_loss is not None:
        summary.extend([
            f'Val Loss: {val_loss:.4f}',
            f'Val Acc: {val_acc:.2f}%',
            f'Precision: {precision:.4f}',
            f'Recall: {recall:.4f}',
            f'F1: {f1:.4f}',
        ])
    print(' | '.join(summary))


## 6. Training pipeline

This cell builds datasets, dataloaders, model, optimizer, scheduler, and runs training with validation metrics based on precision, recall, and F1.

In [ ]:
def main(config_path='E:/Verifixia/Verifixia/Backend/pytorch/config.yaml'):
    """Run the end-to-end training pipeline for image or video deepfake detection."""

    """
    Pipeline overview:
    - Load and sanitize configuration values from YAML.
    - Select CPU or GPU execution and enable mixed precision when CUDA is available.
    - Build the requested dataset type, then either train on the full dataset or create an 80/20 split.
    - Create DataLoaders, model, loss function, optimizer, and cosine annealing scheduler.
    - Train for `num_epochs`, evaluate on validation data, and track precision, recall, and F1.
    - Save the best checkpoint using validation F1 as the model-selection criterion.

    This structure keeps the notebook close to the original script while making each stage of the training flow explicit.
    """
    set_seed(42)
    config = load_config(config_path)
    config['batch_size'] = int(config.get('batch_size', 16))
    config['learning_rate'] = float(config.get('learning_rate', 0.0005))
    config['weight_decay'] = float(config.get('weight_decay', 1e-4))
    config['num_epochs'] = int(config.get('num_epochs', 50))
    config['use_whole_dataset'] = bool(config.get('use_whole_dataset', False))

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    print('=' * 80)
    print('Training run setup')
    print('=' * 80)
    print(f'Config path: {config_path}')
    print(f'Device: {device} | GPU: {gpu_name}')
    print(f'Dataset type: {config["dataset_type"]}')
    print(f'Epochs: {config["num_epochs"]} | Batch size: {config["batch_size"]} | Learning rate: {config["learning_rate"]}')

    if device.type == 'cuda':
        torch.backends.cudnn.benchmark = True

    scaler = GradScaler(enabled=device.type == 'cuda')

    print('Loading dataset...')
    if config['dataset_type'] == 'image':
        full_dataset = ImageDeepfakeDataset(config['train_data_path'])
    else:
        full_dataset = VideoDeepfakeDataset(
            config['train_data_path'],
            frames_per_video=config.get('frames_per_video', 12),
        )

    if config['use_whole_dataset']:
        train_dataset = full_dataset
        val_dataset = None
        cast(Any, train_dataset).transform = get_transforms(config, is_train=True)
    else:
        train_size = int(0.8 * len(full_dataset))
        val_size = len(full_dataset) - train_size
        train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
        cast(Any, train_dataset.dataset).transform = get_transforms(config, is_train=True)
        cast(Any, val_dataset.dataset).transform = get_transforms(config, is_train=False)

    print(f'Total samples: {len(full_dataset)}')
    print(f'Train samples: {len(train_dataset)}')
    print(f'Validation samples: {0 if val_dataset is None else len(val_dataset)}')

    num_workers = config.get('num_workers', 8 if torch.cuda.is_available() else 0)
    pin_memory = device.type == 'cuda'

    train_loader = DataLoader(
        train_dataset,
        batch_size=config['batch_size'],
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=True,
    )

    val_loader = None
    if val_dataset is not None:
        val_loader = DataLoader(
            val_dataset,
            batch_size=config['batch_size'],
            shuffle=False,
            num_workers=num_workers,
            pin_memory=pin_memory,
        )

    print(f'Num workers: {num_workers} | Pin memory: {pin_memory}')
    print('Building model...')
    model = DeepfakeCNNLSTM(
        dataset_type=config['dataset_type'],
        frames_per_video=config.get('frames_per_video', 12),
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay'],
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['num_epochs'])

    print('Optimizer: AdamW | Scheduler: CosineAnnealingLR | Loss: CrossEntropyLoss')
    print('Training starts now...')

    history = initialize_history()
    best_f1 = 0.0
    best_val_acc = 0.0
    patience = config.get('early_stopping_patience', 8) if not config['use_whole_dataset'] else None
    patience_counter = 0

    for epoch in range(config['num_epochs']):
        model.train()
        train_loss = 0.0
        correct = 0.0
        total = 0.0
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{config["num_epochs"]}', leave=False)

        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            if device.type == 'cuda':
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

            train_loss += loss.item()
            _, pred = outputs.max(1)
            total += labels.size(0)
            correct += pred.eq(labels).sum().item()

            running_loss = train_loss / max(1, len(history['epoch']) + 1)
            running_acc = 100.0 * correct / total if total > 0 else 0.0
            progress_bar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{running_acc:.2f}%')

        epoch_train_loss = train_loss / len(train_loader) if len(train_loader) > 0 else 0.0
        train_acc = 100.0 * correct / total if total > 0 else 0.0

        if val_loader is not None:
            model.eval()
            val_loss = 0.0
            val_correct = 0.0
            val_total = 0.0
            all_prec = 0.0
            all_rec = 0.0
            all_f1 = 0.0

            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item()
                    _, pred = outputs.max(1)
                    val_total += labels.size(0)
                    val_correct += pred.eq(labels).sum().item()

                    precision, recall, f1 = compute_metrics(outputs, labels)
                    all_prec += precision * labels.size(0)
                    all_rec += recall * labels.size(0)
                    all_f1 += f1 * labels.size(0)

            epoch_val_loss = val_loss / len(val_loader) if len(val_loader) > 0 else 0.0
            val_acc = 100.0 * val_correct / val_total if val_total > 0 else 0.0
            avg_prec = all_prec / val_total if val_total > 0 else 0.0
            avg_rec = all_rec / val_total if val_total > 0 else 0.0
            avg_f1 = all_f1 / val_total if val_total > 0 else 0.0

            update_history(
                history,
                epoch + 1,
                epoch_train_loss,
                train_acc,
                epoch_val_loss,
                val_acc,
                avg_prec,
                avg_rec,
                avg_f1,
            )
            display_training_dashboard(history, config, has_validation=True)
            print_epoch_summary(epoch + 1, config, epoch_train_loss, train_acc, epoch_val_loss, val_acc, avg_prec, avg_rec, avg_f1)

            if avg_f1 > best_f1:
                best_f1 = avg_f1
                best_val_acc = val_acc
                torch.save(model.state_dict(), config['model_save_path'])
                print(f'Checkpoint updated: {config["model_save_path"]}')
                patience_counter = 0
            else:
                patience_counter += 1
                print(f'No F1 improvement. Early-stop counter: {patience_counter}/{patience}')

            if patience is not None and patience_counter >= patience:
                print('Early stopping triggered because F1 stopped improving.')
                break
        else:
            torch.save(model.state_dict(), config['model_save_path'])
            update_history(history, epoch + 1, epoch_train_loss, train_acc)
            display_training_dashboard(history, config, has_validation=False)
            print_epoch_summary(epoch + 1, config, epoch_train_loss, train_acc)
            print(f'Checkpoint saved: {config["model_save_path"]}')

        scheduler.step()

    print('=' * 80)
    print(f'Training completed. Best F1: {best_f1:.4f} | Best Val Acc: {best_val_acc:.2f}%')
    print('=' * 80)
    return history


## 7. Configure and run

Update the config path if needed, then run the final cell to start training.

In [ ]:
CONFIG_PATH = r'E:/Verifixia/Verifixia/Backend/pytorch/config.yaml'
# Example: CONFIG_PATH = '../Verifixia/Backend/pytorch/config.yaml'

In [9]:
history = main(CONFIG_PATH)
history

FileNotFoundError: [Errno 2] No such file or directory: 'config.yaml'

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import sys

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR / 'Verifixia'
BACKEND_DIR = REPO_ROOT / 'Backend'
INPUT_DIR = REPO_ROOT / 'input'
OUTPUT_DIR = REPO_ROOT / 'output'

if not BACKEND_DIR.exists():
    alt_repo = NOTEBOOK_DIR.parent / 'Verifixia'
    if alt_repo.exists():
        REPO_ROOT = alt_repo
        BACKEND_DIR = REPO_ROOT / 'Backend'
        INPUT_DIR = REPO_ROOT / 'input'
        OUTPUT_DIR = REPO_ROOT / 'output'

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from detection_api import allowed_file, is_video_file, predict_deepfake, predict_deepfake_video

INPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

media_files = sorted(
    path for path in INPUT_DIR.iterdir()
    if path.is_file() and allowed_file(path.name)
)

if not media_files:
    print(f'No supported files found in {INPUT_DIR}')
    print('Add your custom video file there and rerun this cell.')
else:
    run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    summary = {
        'run_timestamp': run_timestamp,
        'input_dir': str(INPUT_DIR),
        'output_dir': str(OUTPUT_DIR),
        'total_files': len(media_files),
        'results': [],
    }

    for file_path in media_files:
        print(f'Analysing {file_path.name} ...')

        if is_video_file(file_path.name):
            prediction, confidence_raw = predict_deepfake_video(str(file_path))
            result = {
                'filename': file_path.name,
                'path': str(file_path),
                'is_video': True,
                'processed_at': datetime.now().isoformat(),
                'prediction': prediction,
                'confidence': round(confidence_raw * 100, 2),
                'confidence_raw': confidence_raw,
                'threat_level': 'high' if confidence_raw > 0.7 else 'medium' if confidence_raw > 0.4 else 'low',
                'model_used': 'Verifixia AI Video Analyser',
                'analysis': {
                    'level': 'Video (frame sampling)',
                    'description': 'Up to 5 evenly-spaced frames extracted and analysed',
                },
            }
        else:
            result = predict_deepfake(str(file_path))
            result = {
                'filename': file_path.name,
                'path': str(file_path),
                'is_video': False,
                'processed_at': datetime.now().isoformat(),
                **result,
                'confidence': round(float(result.get('confidence', 0)), 2),
            }

        summary['results'].append(result)

        per_file_output = OUTPUT_DIR / f'{file_path.stem}_result.json'
        per_file_output.write_text(json.dumps(result, indent=2), encoding='utf-8')

        print(
            f"  -> {result['prediction']} | {result['confidence']:.2f}% | {result.get('model_used', 'unknown model')}"
        )

    summary_output = OUTPUT_DIR / f'custom_input_summary_{run_timestamp}.json'
    summary_output.write_text(json.dumps(summary, indent=2), encoding='utf-8')

    print('')
    print(f'Summary saved to: {summary_output}')
    print(f'Per-file results saved in: {OUTPUT_DIR}')
    summary
